In [ ]:
# Install required packages
!pip install pillow numpy pandas opencv-python pytesseract easyocr matplotlib -q

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import pytesseract
import easyocr

# Configuration
SOURCE_FOLDER = '/kaggle/input/illegible-medical-prescription-images-dataset/data'
DESTINATION_FOLDER = '/kaggle/working/subset'
SUBSET_SIZE = 10
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# Create subset of images
os.makedirs(DESTINATION_FOLDER, exist_ok=True)

all_images = [f for f in os.listdir(SOURCE_FOLDER) if f.endswith('.jpg')]
print(f"Total images available: {len(all_images)}")

if len(all_images) < SUBSET_SIZE:
    raise ValueError(f"Not enough images ({len(all_images)}) to create subset of {SUBSET_SIZE}")

subset_files = random.sample(all_images, SUBSET_SIZE)

for image in subset_files:
    src = os.path.join(SOURCE_FOLDER, image)
    dst = os.path.join(DESTINATION_FOLDER, image)
    shutil.copy2(src, dst)

print(f"Copied {len(subset_files)} images to {DESTINATION_FOLDER}")
print("Files:", subset_files)

In [ ]:
# Load images from subset
image_list = [Image.open(os.path.join(DESTINATION_FOLDER, f)) for f in subset_files]

# Initialize OCR engines
reader = easyocr.Reader(['en'])

print(f"Loaded {len(image_list)} images")
for i, img in enumerate(image_list):
    print(f"  {subset_files[i]}: {img.size} {img.mode}")

In [ ]:
def preprocess_image_pil(image):
    """Simple PIL preprocessing: grayscale + normalize."""
    gray = image.convert('L')
    arr = np.array(gray) / 255.0
    return arr

def preprocess_image_cv(image):
    """OpenCV preprocessing: grayscale + Gaussian blur + Otsu threshold."""
    img_array = np.array(image)
    
    # Handle different input formats
    if len(img_array.shape) == 2:
        img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
    elif img_array.shape[2] == 4:
        img_array = cv2.cvtColor(img_array, cv2.COLOR_RGBA2RGB)

    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return Image.fromarray(thresh)

In [ ]:
# Preprocess images using both methods
preprocessed_pil = [preprocess_image_pil(img) for img in image_list]
preprocessed_cv = [preprocess_image_cv(img) for img in image_list]

print(f"PIL preprocessed: {len(preprocessed_pil)} images")
print(f"CV preprocessed: {len(preprocessed_cv)} images")

In [ ]:
# Extract text with pytesseract (using PIL preprocessing)
texts_tesseract = [
    pytesseract.image_to_string(Image.fromarray((img * 255).astype(np.uint8)), config='--psm 6')
    for img in preprocessed_pil
]

In [ ]:
# Extract text with EasyOCR (using CV preprocessing)
texts_easyocr = [
    ' '.join(reader.readtext(np.array(img), detail=0))
    for img in preprocessed_cv
]

In [ ]:
# Display results
for i, (fname, pil_img, cv_img, tess_text, easy_text) in enumerate(
    zip(subset_files, preprocessed_pil, preprocessed_cv, texts_tesseract, texts_easyocr)
):
    print(f"{'='*60}")
    print(f"Image {i+1}: {fname}")
    print(f"{'='*60}")
    
    # Show PIL preprocessed
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(pil_img, cmap='gray')
    plt.title(f'PIL Preprocessed')
    plt.axis('off')
    
    # Show CV preprocessed
    plt.subplot(1, 2, 2)
    plt.imshow(np.array(cv_img), cmap='gray')
    plt.title(f'CV Preprocessed')
    plt.axis('off')
    plt.show()
    
    print(f"Tesseract:\n{tess_text.strip() or '(empty)'}")
    print(f"EasyOCR:\n{easy_text.strip() or '(empty)'}")
    print()

In [ ]:
# Create DataFrame with results
df = pd.DataFrame({
    'Image Name': subset_files,
    'Tesseract Text': [t.strip() for t in texts_tesseract],
    'EasyOCR Text': [t.strip() for t in texts_easyocr]
})

print(df.to_string(index=False))

# Save to CSV
df.to_csv('extracted_texts.csv', index=False)
print("\nSaved to extracted_texts.csv")